In [1]:
%pip install -U youtube-transcript-api langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers python-dotenv

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)

   ---------------------------------------- 0/2 [defusedxml]
   ---------------------------------------- 0/2 [defusedxml]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   ---------------------------------------- 2/2 [youtube-transcript-api]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
       raise ValueError("GROQ_API_KEY not found. Add it to your .env file and restart the notebook kernel.")

print("Environment is ready.")

C:\Users\aksha\AppData\Local\Temp\ipykernel_64128\2041862891.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Environment is ready.


In [5]:
# 1. Provide the YouTube video ID or URL
# Example: For https://www.youtube.com/watch?v=kCc8FmEb1nY, video_id is "kCc8FmEb1nY"
video_url_or_id = "kCc8FmEb1nY"  # <-- Replace with your YouTube video ID or URL

# Extract clean video ID if a full URL was provided
if "v=" in video_url_or_id:
    video_id = video_url_or_id.split("v=")[1].split("&")[0]
elif "youtu.be/" in video_url_or_id:
    video_id = video_url_or_id.split("youtu.be/")[1].split("?")[0]
else:
    video_id = video_url_or_id.strip()

# 2. Fetch transcript using YouTubeTranscriptApi instance
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id)

# 3. Convert transcript snippets into a list of dictionaries
transcript_list = fetched_transcript.to_raw_data()

# 4. Join transcript text into one string
transcript = " ".join(chunk["text"] for chunk in transcript_list)
print(f"Transcript fetched successfully! (Total length: {len(transcript)} characters)\n")
print("Preview:", transcript[:400] + "...")


Transcript fetched successfully! (Total length: 108990 characters)

Preview: hi everyone so by now you have probably heard of chat GPT it has taken the world and AI Community by storm and it is a system that allows you to interact with an AI and give it text based tasks so for example we can ask chat GPT to write us a small Hau about how important it is that people understand Ai and then they can use it to improve the world and make it more prosperous so when we run this A...


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

video_id = "Gfr50f6ZBvo"  # YouTube video ID

try:
    # Fetch the transcript (v1.2+ uses instance method .fetch())
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript_list = fetched_transcript.to_raw_data()

    # Convert transcript snippets into one string
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(f"Transcript fetched successfully! ({len(transcript)} characters)\n")
    print("Preview:\n", transcript[:500] + "...")

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No English transcript found for this video.")
except Exception as e:
    print(f"An error occurred: {e}")


Transcript fetched successfully! (133836 characters)

Preview:
 the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful ...


In [9]:
# Check that a transcript was fetched
print("Transcript characters:", len(transcript))

Transcript characters: 133836


Step 1b — Indexing (Text Splitting)

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript]) if transcript else []
print("Number of chunks:", len(chunks))

Number of chunks: 168


In [11]:
# Display one chunk if it exists
if len(chunks) > 100:
    print(chunks[100])
elif chunks:
    print(chunks[0])
else:
    print("No chunks available. Check whether the video has an accessible transcript.")

page_content='and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask what's involved in that

Step 1c & 1d — Embedding Generation and Storing in Vector Store
Groq is used for text generation, not embeddings. Hugging Face creates the document/query vectors locally.

In [12]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if not chunks:
    raise ValueError("No transcript chunks to index. Resolve transcript retrieval before continuing.")

vector_store = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store created.")

c:\Users\aksha\OneDrive\Desktop\langChain models\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1159.58it/s]


FAISS vector store created.


In [14]:
# View the FAISS internal document-ID mapping
vector_store.index_to_docstore_id

{0: 'bb3dbe41-3ece-49e5-8167-74f1d7034bbf',
 1: '8c2e7e29-7d78-4675-8155-30536b9b85d6',
 2: '097031ce-4e3e-4c6a-a1b4-0b5e025146cd',
 3: 'fa8d8cfe-0edb-49ea-b030-c84ce65644dc',
 4: '5b68e832-d5a5-4123-a3ff-0333b8bfa817',
 5: '9e2d0dd6-974d-4ef5-b116-2e7f7dbd730d',
 6: 'cfc8a269-6423-4617-8b08-71c7d4d7d5f0',
 7: '5d362c5e-c5d4-4449-abcb-655a1b13ebc4',
 8: '684729bd-b671-42c3-8dd6-2ff166f56737',
 9: 'ced27a78-278e-4b31-8751-9376ebe5fd94',
 10: 'f06c34c6-1dbd-4c61-80cd-16941ba8ed3b',
 11: '76008676-436a-450a-aad9-82bab6ba2892',
 12: 'df9787e0-1898-4b05-ac0c-2588595989b7',
 13: 'abb6d77b-a91b-4a4d-9804-ff5322b03076',
 14: '53410200-8944-48d8-9ee2-5bd17e2b113b',
 15: '0ebd2763-a078-48da-82e1-4fc0a1c142e4',
 16: 'c67bf937-6023-451f-bcb5-4bf0e480521a',
 17: '00ad7615-feb0-4118-8fcf-f6d8683fce31',
 18: 'a1a90602-a011-4b1f-8fa4-495f61a5adf2',
 19: '4a43e924-e9af-4e2f-8e6f-c4e671696583',
 20: '7c9b021d-b915-4507-bf7a-94a38f5ba962',
 21: '5852d1e0-343c-405a-97f5-483015f44ad7',
 22: 'cd226207-1153-

In [15]:
# Retrieve one stored document using an ID generated by this vector store
sample_id = next(iter(vector_store.index_to_docstore_id.values()))
vector_store.get_by_ids([sample_id])

[Document(id='bb3dbe41-3ece-49e5-8167-74f1d7034bbf', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

Step 2 — Retrieval

In [16]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002C861FBABA0>, search_kwargs={'k': 4})

In [17]:
retriever.invoke("What is DeepMind?")

[Document(id='a712d944-8550-49af-b417-6adf680e8d90', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

Step 3 — Augmentation

In [18]:
# Groq chat model
# Model ID is listed in Groq's current supported-model catalogue.
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

In [19]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}

Question: {question}
""",
    input_variables=["context", "question"]
)

In [20]:
question = "Is the topic of nuclear fusion discussed in this video? If yes, what was discussed?"
retrieved_docs = retriever.invoke(question)

In [21]:
retrieved_docs

[Document(id='a501d541-478a-4aa9-bbe6-78ff1c157947', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [22]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
print(context_text)

in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of

th

In [23]:
final_prompt = prompt.invoke({
    "context": context_text,
    "question": question
})

final_prompt

StringPromptValue(text="\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don't know.\n\nin this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which o

Step 4 — Generation

In [24]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes. The video discusses nuclear fusion. It describes how the team partnered with EPFL’s test reactor to explore fusion bottlenecks, used AI (including deep reinforcement learning) to develop controllers that can shape, contain, and hold plasma for record times, and examined which physics, materials‑science, and engineering challenges are amenable to AI methods. It also mentions a Nature paper on controlling plasma shapes, talks about future collaborations with fusion startups, and references a paper on “magnetic control of tokamak plasmas with deep reinforcement learning.”


Building a RAG Chain

In [25]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser

In [26]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [27]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [28]:
parallel_chain.invoke("Who is Demis?")

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [29]:
parser = StrOutputParser()

In [30]:
main_chain = parallel_chain | prompt | llm | parser

In [31]:
main_chain.invoke("Can you summarize the video?")

'The video is a loosely‑structured conversation that drifts through several big‑picture ideas:\n\n* It opens with a reflection on how a deeper, simpler theory of physics—one that goes beyond the Standard Model—might eventually explain long‑standing mysteries such as consciousness, life and gravity.  \n* The speakers praise the ability to explain complex topics in simple terms (citing Richard\u202fFeynman) as a hallmark of true understanding, and they discuss how humans are already symbiotic with computers, hinting at future enhancement.  \n* A brief historical digression follows, recalling Claude\u202fShannon’s 1949 chess program, Turing’s hand‑run chess algorithm, and the landmark Deep\u202fBlue versus Kasparov match, noting that the human mind’s elegance impressed the speaker more than the raw computing power.  \n* Interspersed are acknowledgments to the podcast host (Demas), a quote from Edskar\u202fDykstra about computer science being “no more about computers than the music,” and a